# 乳腺肿瘤分类练习

使用 scikit-learn 附带的 Wisconsin Diagnostic Breast Cancer 数据，对比 TensorFlow 的逻辑回归与两层全连接神经网络。实验把恶性编码为 1，统计遗漏的恶性样本，并比较两个模型的学习曲线。

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("TensorFlow:", tf.__version__)
print("可用 GPU:", tf.config.list_physical_devices("GPU"))

## 数据、标签和分层划分

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)
print("数据集类型：", type(cancer))
print("可用字段：", cancer.keys())
print("特征类型：", type(cancer.data))
print("标签类型：", type(cancer.target))
print("特征 shape：", cancer.data.shape)
print("标签 shape：", cancer.target.shape)
print("标签名称：", cancer.target_names)
display(cancer.data.head())
display(cancer.target.value_counts().sort_index())

In [ ]:
X = cancer.data.copy()
y = (cancer.target == 0).astype("int32")

print("X：", type(X), X.shape)
print("X 的数据类型：", X.dtypes.unique())
print("y：", type(y), y.shape, y.dtype)
print("模型标签计数：\n", y.value_counts().sort_index())
print("X 和 y 索引对齐：", X.index.equals(y.index))
display(pd.DataFrame({"原始标签": cancer.target.head(),
                      "模型标签": y.head()}))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

for name, X_part, y_part in [
    ("训练集", X_train, y_train),
    ("验证集", X_val, y_val),
    ("测试集", X_test, y_test),
]:
    print(name, X_part.shape, y_part.shape,
          y_part.value_counts().sort_index().to_dict())

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_val_scaled = scaler.transform(X_val).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

y_train_tf = y_train.to_numpy(dtype="float32").reshape(-1, 1)
y_val_tf = y_val.to_numpy(dtype="float32").reshape(-1, 1)
y_test_tf = y_test.to_numpy(dtype="float32").reshape(-1, 1)

print("训练：", type(X_train_scaled), X_train_scaled.shape, X_train_scaled.dtype, y_train_tf.shape, y_train_tf.dtype)
print("验证：", X_val_scaled.shape, y_val_tf.shape)
print("测试：", X_test_scaled.shape, y_test_tf.shape)
print("训练集前三列均值/标准差：", np.round(X_train_scaled.mean(axis=0)[:3], 4), np.round(X_train_scaled.std(axis=0)[:3], 4))

## 逻辑回归与损失曲线

In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)
n_features = X_train_scaled.shape[1]

logistic_model = tf.keras.Sequential([
    tf.keras.Input(shape=(n_features,), name="features"),
    tf.keras.layers.Dense(1, activation=None, name="logit")
], name="logistic_regression")

logistic_model.summary()
W, b = logistic_model.layers[0].get_weights()
print("W shape：", W.shape)
print("b shape：", b.shape)

In [ ]:
loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
train_accuracy = tf.keras.metrics.BinaryAccuracy(
    threshold=0.0, name="accuracy"
)

logistic_model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=[train_accuracy]
)

print("compile 完成，模型尚未训练")

In [ ]:
params_before = [p.copy() for p in logistic_model.get_weights()]

history_lr = logistic_model.fit(
    X_train_scaled, y_train_tf,
    validation_data=(X_val_scaled, y_val_tf),
    epochs=100,
    batch_size=32,
    verbose=0
)

params_after = logistic_model.get_weights()
print("History 类型：", type(history_lr))
print("记录字段：", history_lr.history.keys())
print("W、b 是否更新：", [not np.allclose(a, b) for a, b in zip(params_before, params_after)])
print("最终训练 loss/accuracy：", history_lr.history["loss"][-1], history_lr.history["accuracy"][-1])
print("最终验证 loss/accuracy：", history_lr.history["val_loss"][-1], history_lr.history["val_accuracy"][-1])

In [ ]:
history_df = pd.DataFrame(history_lr.history)
epochs = np.arange(1, len(history_df) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history_df["loss"], label="train")
axes[0].plot(epochs, history_df["val_loss"], label="validation")
axes[0].set(xlabel="Epoch", ylabel="Binary cross-entropy", title="Loss curve")
axes[0].legend()

axes[1].plot(epochs, history_df["accuracy"], label="train")
axes[1].plot(epochs, history_df["val_accuracy"], label="validation")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Accuracy curve")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
lr_test_logits = logistic_model.predict(X_test_scaled, verbose=0)
lr_test_prob = tf.sigmoid(lr_test_logits).numpy()
lr_test_pred = (lr_test_prob >= 0.5).astype("int32")

print("logits：", type(lr_test_logits), lr_test_logits.shape, lr_test_logits[:5].ravel())
print("概率：", type(lr_test_prob), lr_test_prob.shape, lr_test_prob[:5].ravel())
print("两种阈值结果一致：", np.array_equal(lr_test_pred, (lr_test_logits >= 0.0).astype("int32")))

preview = pd.DataFrame({
    "真实标签": y_test_tf[:5].ravel().astype("int32"),
    "logit": lr_test_logits[:5].ravel(),
    "恶性概率": lr_test_prob[:5].ravel(),
    "预测标签": lr_test_pred[:5].ravel()
})
display(preview)

In [ ]:
from sklearn import metrics

y_true = y_test_tf.ravel().astype("int32")
y_pred = lr_test_pred.ravel()
y_prob = lr_test_prob.ravel()

scores = pd.Series({"Accuracy": metrics.accuracy_score(y_true, y_pred),
                    "Precision": metrics.precision_score(y_true, y_pred),
                    "Recall": metrics.recall_score(y_true, y_pred),
                    "F1": metrics.f1_score(y_true, y_pred),
                    "ROC-AUC": metrics.roc_auc_score(y_true, y_prob)})
print(scores.round(4))

cm = metrics.confusion_matrix(y_true, y_pred, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred benign", "Pred malignant"], yticklabels=["True benign", "True malignant"])
plt.show()

## 一例假阴性及其特征贡献

In [ ]:
fn_mask = (y_true == 1) & (y_pred == 0)
fn_pos = np.flatnonzero(fn_mask)
fn_indices = X_test.index[fn_pos]
print("FN数量、测试集位置、原始索引：", len(fn_pos), fn_pos, fn_indices.tolist())
print("logit、恶性概率：", lr_test_logits.ravel()[fn_pos], y_prob[fn_pos])

if len(fn_pos) > 0:
    W_trained, b_trained = logistic_model.layers[0].get_weights()
    W_trained = W_trained.ravel()
    contribution = X_test_scaled[fn_pos[0]] * W_trained
    print("重构 logit：", contribution.sum() + b_trained[0])
    feature_effects = pd.DataFrame(
        {"raw": X_test.iloc[fn_pos[0]].to_numpy(), "scaled": X_test_scaled[fn_pos[0]], "weight": W_trained, "x*w": contribution}, index=X_test.columns)
    extremes = pd.concat([feature_effects.nsmallest(5, "x*w"), feature_effects.nlargest(5, "x*w")])
    display(extremes)

## 小型神经网络与模型对比

In [ ]:
tf.keras.utils.set_random_seed(42)

nn_model = tf.keras.Sequential([
    tf.keras.Input(shape=(30,), name="features"),
    tf.keras.layers.Dense(16, activation="relu", name="hidden_1"),
    tf.keras.layers.Dense(8, activation="relu", name="hidden_2"),
    tf.keras.layers.Dense(1, activation=None, name="logit")
], name="small_neural_network")

nn_model.summary()
for layer in nn_model.layers:
    print(layer.name, [w.shape for w in layer.get_weights()])

In [ ]:
nn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.BinaryAccuracy(threshold=0.0, name="accuracy")]
)

history_nn = nn_model.fit(
    X_train_scaled, y_train_tf,
    validation_data=(X_val_scaled, y_val_tf),
    epochs=100,
    batch_size=32,
    verbose=0
)

print("最终训练 loss/accuracy：", history_nn.history["loss"][-1], history_nn.history["accuracy"][-1])
print("最终验证 loss/accuracy：", history_nn.history["val_loss"][-1], history_nn.history["val_accuracy"][-1])

In [ ]:
nn_best_epoch = np.argmin(history_nn.history["val_loss"]) + 1
print("神经网络最低验证 loss 的 epoch：", nn_best_epoch)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in [("LogReg", history_lr), ("NeuralNet", history_nn)]:
    e = np.arange(1, len(h.history["loss"]) + 1)
    axes[0].plot(e, h.history["loss"], label=f"{name} train")
    axes[0].plot(e, h.history["val_loss"], label=f"{name} validation")
    axes[1].plot(e, h.history["accuracy"], label=f"{name} train")
    axes[1].plot(e, h.history["val_accuracy"], label=f"{name} validation")
axes[0].set(title="Loss comparison", xlabel="Epoch", ylabel="BCE")
axes[1].set(title="Accuracy comparison", xlabel="Epoch", ylabel="Accuracy")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
nn_test_logits = nn_model.predict(X_test_scaled, verbose=0)
nn_test_prob = tf.sigmoid(nn_test_logits).numpy().ravel()
nn_test_pred = (nn_test_prob >= 0.5).astype("int32")

nn_scores = pd.Series({
    "Accuracy": metrics.accuracy_score(y_true, nn_test_pred),
    "Precision": metrics.precision_score(y_true, nn_test_pred),
    "Recall": metrics.recall_score(y_true, nn_test_pred),
    "F1": metrics.f1_score(y_true, nn_test_pred),
    "ROC-AUC": metrics.roc_auc_score(y_true, nn_test_prob)
})
comparison = pd.DataFrame({"Logistic Regression": scores, "Neural Network": nn_scores})
display(comparison.round(4))
nn_cm = metrics.confusion_matrix(y_true, nn_test_pred, labels=[0, 1])
print("神经网络混淆矩阵：\n", nn_cm)